# VOICECLONE-QC RVC Bridge

Run the Drive authorization cell immediately after configuration. This notebook is pinned to the validated RVC v2 / 40k / RMVPE pipeline and saves resumable checkpoints to Drive.


In [ ]:
#@title 1. Configuration VOICECLONE-QC v1.2.0
from pathlib import Path

NOTEBOOK_VERSION = "1.2.0"
MODEL_NAME = "Alertes_Stephanie"  #@param {type:"string"}
RUN_MODE = "new"  #@param ["new", "resume"]
TARGET_SAMPLE_RATE = "40k"
MODEL_ARCHITECTURE = "v2"
PRETRAIN_TYPE = "OV2"
PITCH_METHOD = "rmvpe"
TOTAL_EPOCHS = 200  #@param {type:"integer"}
SAVE_FREQUENCY = 10  #@param {type:"integer"}
BATCH_SIZE = 7  #@param {type:"integer"}
CHECKPOINT_SYNC_SECONDS = 120  #@param {type:"integer"}

# Pin the exact RVC code and model assets used by this notebook.
RVC_REPOSITORY_COMMIT = "d618280cdef162c39bf74d03362592db5c41ad80"
TORCHCREPE_COMMIT = "19e2ec3d494c0797a5ff2a11408ec5838fba6681"
ORVC_REVISION = "425f8006582161af571e0d7f5ce646a535a14b65"

DRIVE_DATASET_DIR = "/content/drive/MyDrive/VOICECLONE_Datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RVC_Output"
DRIVE_TRAINING_DIR = "/content/drive/MyDrive/VOICECLONE_Training"
DRIVE_RUN_DIR = Path(DRIVE_TRAINING_DIR) / MODEL_NAME
DRIVE_EXPERIMENT_DIR = DRIVE_RUN_DIR / "experiment"
DRIVE_CHECKPOINT_DIR = DRIVE_RUN_DIR / "checkpoints"

ZIP_PATH = f"{DRIVE_DATASET_DIR}/{MODEL_NAME}_dataset_cleaned.zip"
NOW_DIR = "/content/Mangio-RVC-Fork"
EXP_DIR = f"{NOW_DIR}/logs/{MODEL_NAME}"
DATASET_DIR = f"/content/voiceclone_qc/{MODEL_NAME}/dataset"

if not MODEL_NAME.strip():
    raise ValueError("MODEL_NAME cannot be empty.")
if RUN_MODE not in {"new", "resume"}:
    raise ValueError("RUN_MODE must be 'new' or 'resume'.")
if (TARGET_SAMPLE_RATE, MODEL_ARCHITECTURE, PRETRAIN_TYPE) != ("40k", "v2", "OV2"):
    raise ValueError("This validated notebook is pinned to 40k, v2 and OV2 assets.")
if PITCH_METHOD != "rmvpe":
    raise ValueError("Use RMVPE to keep the production quality setting consistent.")
if TOTAL_EPOCHS < 1 or SAVE_FREQUENCY < 1 or BATCH_SIZE < 1:
    raise ValueError("Training values must be positive integers.")

print(
    f"VOICECLONE-QC RVC Bridge v{NOTEBOOK_VERSION} | "
    f"Model: {MODEL_NAME} | mode: {RUN_MODE} | target epochs: {TOTAL_EPOCHS}"
)


In [ ]:
#@title 2. Mount Google Drive - run this immediately
import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

for directory in (DRIVE_DATASET_DIR, DRIVE_OUTPUT_DIR, DRIVE_TRAINING_DIR, DRIVE_RUN_DIR):
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Google Drive is ready. You can now leave Colab to continue the setup.")


In [ ]:
#@title 3. Install Dependencies
import subprocess
import sys

system_packages = [
    "build-essential",
    "python3-dev",
    "ffmpeg",
    "aria2",
]

python_packages = [
    "faiss-cpu",
    "ffmpeg-python",
    "praat-parselmouth",
    "pyworld",
    "numpy",
    "numba",
    "librosa",
    "tensorboardX",
    "tensorboard",
    "onnx",
    "onnxruntime-gpu",
    "torchcrepe",
    "python-dotenv",
    "av",
    "scikit-learn",
]

def run_command(command, label):
    print(f"\nInstalling: {label}", flush=True)
    try:
        subprocess.check_call(command)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f"\nINSTALLATION FAILED: {label}\n"
            f"Command: {' '.join(command)}\n"
            "Copy the error shown immediately above this message."
        ) from error

print("Updating package list...", flush=True)
run_command(["apt-get", "update", "-qq"], "system package list")

print("Installing system packages...", flush=True)
for package in system_packages:
    run_command(
        ["apt-get", "install", "-qq", "-y", package],
        f"system package {package}",
    )

print("\nUpdating pip tools...", flush=True)
run_command(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    "pip, setuptools and wheel",
)

print("\nInstalling Python packages...", flush=True)
for package in python_packages:
    run_command(
        [sys.executable, "-m", "pip", "install", "--upgrade", package],
        f"Python package {package}",
    )

print("\nInstalling fairseq-fixed...", flush=True)
run_command(
    [sys.executable, "-m", "pip", "install", "fairseq-fixed"],
    "fairseq-fixed",
)

print("\nDependencies ready.", flush=True)

In [ ]:
#@title 4. Clone RVC Repository (pinned)
import os
import shutil
import subprocess

repo_dir = Path("/content/Mangio-RVC-Fork")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
torchcrepe_dir = Path("/content/torchcrepe")
if torchcrepe_dir.exists():
    shutil.rmtree(torchcrepe_dir)

subprocess.check_call([
    "git", "clone", "https://github.com/Mangio621/Mangio-RVC-Fork.git", str(repo_dir)
])
subprocess.check_call(["git", "-C", str(repo_dir), "checkout", RVC_REPOSITORY_COMMIT])
subprocess.check_call([
    "git", "clone", "https://github.com/maxrmorrison/torchcrepe.git", str(torchcrepe_dir)
])
subprocess.check_call(["git", "-C", str(torchcrepe_dir), "checkout", TORCHCREPE_COMMIT])
shutil.copytree(torchcrepe_dir / "torchcrepe", repo_dir / "torchcrepe", dirs_exist_ok=True)
os.chdir(repo_dir)
print(f"Repository ready: {repo_dir} @ {RVC_REPOSITORY_COMMIT[:12]}")


In [ ]:
#@title 5. GPU Check
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Change runtime type to GPU.')

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory // 1024 // 1024, 'MB')

gpus = '-'.join(str(i) for i in range(torch.cuda.device_count()))
print('Using GPU ids:', gpus)


In [ ]:
#@title 6. Download verified pretrained models and RMVPE
import hashlib
import json
import subprocess

assets = {
    Path(NOW_DIR) / "pretrained_v2" / "f0G40k_OV2.pth": [
        f"https://huggingface.co/ORVC/Ov2Super/resolve/{ORVC_REVISION}/f0Ov2Super40kG.pth",
        "https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kG.pth",
    ],
    Path(NOW_DIR) / "pretrained_v2" / "f0D40k_OV2.pth": [
        f"https://huggingface.co/ORVC/Ov2Super/resolve/{ORVC_REVISION}/f0Ov2Super40kD.pth",
        "https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kD.pth",
    ],
    Path(NOW_DIR) / "hubert_base.pt": [
        "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt",
    ],
    Path(NOW_DIR) / "rmvpe.pt": [
        "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt",
    ],
    Path(NOW_DIR) / "configs" / "40k.json": [
        f"https://raw.githubusercontent.com/Mangio621/Mangio-RVC-Fork/{RVC_REPOSITORY_COMMIT}/configs/40k.json",
    ],
}

def download_verified(destination, urls):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    failures = []
    for url in urls:
        if temporary.exists():
            temporary.unlink()
        try:
            subprocess.check_call([
                "curl", "--fail", "--location", "--retry", "5", "--retry-all-errors",
                "--connect-timeout", "30", "--output", str(temporary), url,
            ])
            if temporary.stat().st_size < 100:
                raise RuntimeError("downloaded file is unexpectedly small")
            temporary.replace(destination)
            return url, hashlib.sha256(destination.read_bytes()).hexdigest()
        except Exception as error:
            failures.append(f"{url}: {error}")
    raise RuntimeError(
        f"Unable to download {destination.name}. Tried:\n" + "\n".join(failures)
    )

manifest = {}
for destination, urls in assets.items():
    used_url, checksum = download_verified(destination, urls)
    manifest[str(destination.relative_to(NOW_DIR))] = {
        "url": used_url,
        "sha256": checksum,
        "bytes": destination.stat().st_size,
    }

manifest_path = Path(NOW_DIR) / "voiceclone_asset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Verified {len(manifest)} assets: {manifest_path}")


In [ ]:
#@title 7. Restore a resumable training run from Drive
import shutil

local_experiment = Path(EXP_DIR)
if RUN_MODE == "resume":
    if not DRIVE_EXPERIMENT_DIR.exists():
        raise FileNotFoundError(
            f"No Drive checkpoint backup found for {MODEL_NAME}: {DRIVE_EXPERIMENT_DIR}"
        )
    if local_experiment.exists():
        shutil.rmtree(local_experiment)
    shutil.copytree(DRIVE_EXPERIMENT_DIR, local_experiment)
    print(f"Restored experiment: {DRIVE_EXPERIMENT_DIR} -> {local_experiment}")
else:
    if local_experiment.exists():
        shutil.rmtree(local_experiment)
    print("New run selected. Existing Drive backups are preserved until this run creates new ones.")


In [ ]:
#@title 8. Load VOICECLONE-QC Dataset ZIP
import os
import shutil
import zipfile

if RUN_MODE == "resume":
    print("Resume mode: dataset import is skipped.")
else:
    dataset_dir = Path(DATASET_DIR)
    if not Path(ZIP_PATH).exists():
        raise FileNotFoundError(f"Dataset ZIP not found: {ZIP_PATH}")
    if dataset_dir.parent.exists():
        shutil.rmtree(dataset_dir.parent)
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(dataset_dir.parent)
    readme = dataset_dir / "README.txt"
    if readme.exists():
        readme.unlink()
    wav_files = sorted(dataset_dir.glob("*.wav"))
    if not wav_files:
        raise RuntimeError(f"No WAV files found in {dataset_dir}")
    print(f"Dataset ready: {dataset_dir} ({len(wav_files)} WAV files)")


In [ ]:
#@title 9. Setup CSVDB
import os
import csv

os.chdir(NOW_DIR)
os.makedirs('csvdb', exist_ok=True)

with open('csvdb/formanting.csv', 'w', newline='') as frmnt:
    csv.writer(frmnt, delimiter=',').writerow([False, 1.0, 1.0])

with open('csvdb/stop.csv', 'w', newline='') as stp:
    csv.writer(stp, delimiter=',').writerow([False])

DoFormant, Quefrency, Timbre = False, 1.0, 1.0
print('CSVDB ready:', os.path.abspath('csvdb/formanting.csv'))


In [ ]:
#@title 10. Preprocess Dataset
import os
import subprocess

if RUN_MODE == "resume":
    print("Resume mode: preprocessing is skipped.")
else:
    cpu_threads = min(2, os.cpu_count() or 1)
    Path(EXP_DIR).mkdir(parents=True, exist_ok=True)
    command = [
        "python", "trainset_preprocess_pipeline_print.py", DATASET_DIR, "40000",
        str(cpu_threads), EXP_DIR, "1",
    ]
    print(" ".join(command))
    subprocess.check_call(command)
    print("Preprocessing complete.")


In [ ]:
#@title 11. Python 3.12 / PyTorch Compatibility Patch
from pathlib import Path

sitecustomize = Path(NOW_DIR) / "sitecustomize.py"
sitecustomize.write_text(r'''
import pkgutil
import importlib.machinery

if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def _voiceclone_find_module(self, fullname, path=None):
        spec = self.find_spec(fullname)
        return None if spec is None else spec.loader
    importlib.machinery.FileFinder.find_module = _voiceclone_find_module

if not hasattr(pkgutil, "ImpImporter"):
    class ImpImporter:
        def __init__(self, *args, **kwargs):
            pass
        def find_module(self, fullname, path=None):
            return None
    pkgutil.ImpImporter = ImpImporter

if not hasattr(pkgutil, "ImpLoader"):
    class ImpLoader:
        pass
    pkgutil.ImpLoader = ImpLoader

try:
    import torch
    _voiceclone_original_torch_load = torch.load

    def _voiceclone_torch_load_compat(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _voiceclone_original_torch_load(*args, **kwargs)

    torch.load = _voiceclone_torch_load_compat
except Exception:
    pass
''', encoding="utf-8")

print("Compatibility patch written:", sitecustomize)


In [ ]:
#@title 12. Feature Extraction RMVPE
import os
import subprocess

if RUN_MODE == "resume":
    print("Resume mode: RMVPE and HuBERT extraction are skipped.")
else:
    cpu_threads = min(2, os.cpu_count() or 1)
    environment = os.environ.copy()
    environment["PYTHONPATH"] = f"{NOW_DIR}:{environment.get('PYTHONPATH', '')}"
    f0_command = ["python", "extract_f0_print.py", EXP_DIR, str(cpu_threads), "rmvpe", "128"]
    feature_command = ["python", "extract_feature_print.py", "device", "1", "0", "0", EXP_DIR, "v2"]
    print(" ".join(f0_command))
    subprocess.check_call(f0_command, env=environment)
    print(" ".join(feature_command))
    subprocess.check_call(feature_command, env=environment)
    print("Feature extraction complete.")


In [ ]:
#@title 12b. Matplotlib / NumPy Compatibility Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

if "def _voiceclone_canvas_tostring_rgb" not in text:
    marker = "import matplotlib.pyplot as plt\n"
    patch = '''import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

if not hasattr(FigureCanvasAgg, "tostring_rgb"):
    def _voiceclone_canvas_tostring_rgb(self):
        return self.buffer_rgba().tobytes()
    FigureCanvasAgg.tostring_rgb = _voiceclone_canvas_tostring_rgb
'''
    if marker in text:
        text = text.replace(marker, patch, 1)
    else:
        text = patch + "\n" + text

text = text.replace(
    'np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")',
    'np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)'
)

text = text.replace(
    "np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep='')",
    "np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)"
)

utils_path.write_text(text, encoding="utf-8")
print("Patched:", utils_path)


In [ ]:
#@title 12c. RGB Canvas Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

text = text.replace(
    "return self.buffer_rgba().tobytes()",
    "import numpy as _np\n        return _np.asarray(self.buffer_rgba())[:, :, :3].tobytes()"
)

utils_path.write_text(text, encoding="utf-8")
print("RGB patch OK:", utils_path)


In [ ]:
#@title 13. Save preprocessed data and checkpoints to Drive
import shutil
import time

def copy_file_atomically(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    shutil.copy2(source, temporary)
    temporary.replace(destination)

def sync_checkpoints_to_drive():
    local = Path(EXP_DIR)
    copied = []
    for pattern in ("G_*.pth", "D_*.pth", "filelist.txt", "config.json"):
        for item in local.glob(pattern):
            copy_file_atomically(item, DRIVE_CHECKPOINT_DIR / item.name)
            copied.append(item.name)
    if copied:
        print(f"Checkpoint backup: {', '.join(sorted(copied))}")
    return copied

def sync_experiment_to_drive():
    local = Path(EXP_DIR)
    if not local.exists():
        raise FileNotFoundError(f"Experiment is missing: {local}")
    staging = DRIVE_RUN_DIR / "experiment_staging"
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(local, staging)
    if DRIVE_EXPERIMENT_DIR.exists():
        shutil.rmtree(DRIVE_EXPERIMENT_DIR)
    staging.replace(DRIVE_EXPERIMENT_DIR)
    sync_checkpoints_to_drive()
    print(f"Full experiment backup complete: {DRIVE_EXPERIMENT_DIR}")

if RUN_MODE == "new":
    sync_experiment_to_drive()
else:
    print("Resume mode: the restored backup remains the source of truth.")


In [ ]:
#@title 14. Train RVC Model with automatic checkpoint backups
import os
import subprocess
import time

os.chdir(NOW_DIR)
sync_checkpoints_to_drive()
environment = os.environ.copy()
environment["PYTHONPATH"] = f"{NOW_DIR}:{environment.get('PYTHONPATH', '')}"
command = [
    "python", "train_nsf_sim_cache_sid_load_pretrain.py",
    "-e", MODEL_NAME, "-sr", "40k", "-f0", "1", "-bs", str(BATCH_SIZE),
    "-g", "0", "-te", str(TOTAL_EPOCHS), "-se", str(SAVE_FREQUENCY),
    "-pg", "pretrained_v2/f0G40k_OV2.pth", "-pd", "pretrained_v2/f0D40k_OV2.pth",
    "-l", "1", "-c", "0", "-sw", "1", "-v", "v2", "-li", "3",
]
print(" ".join(command))
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=environment,
)
last_sync = time.monotonic()
for line in process.stdout:
    print(line, end="")
    if time.monotonic() - last_sync >= CHECKPOINT_SYNC_SECONDS:
        sync_checkpoints_to_drive()
        last_sync = time.monotonic()
exit_code = process.wait()
sync_checkpoints_to_drive()
if exit_code != 0:
    raise RuntimeError(f"Training stopped with exit code {exit_code}. Checkpoints remain on Drive.")
sync_experiment_to_drive()
print("Training complete and fully backed up to Drive.")


In [ ]:
#@title 15. Train Index
import os
import sys
import traceback
import numpy as np
import faiss

now_dir = NOW_DIR
experiment_name = MODEL_NAME
model_architecture = MODEL_ARCHITECTURE
exp_dir = EXP_DIR
feature_dir = f'{exp_dir}/3_feature256' if model_architecture == 'v1' else f'{exp_dir}/3_feature768'

if not os.path.exists(feature_dir):
    raise Exception('No features exist. Run Feature Extraction first.')
files = sorted(os.listdir(feature_dir))
if not files:
    raise Exception('No feature files found. Run Feature Extraction first.')

try:
    from sklearn.cluster import MiniBatchKMeans
except Exception:
    MiniBatchKMeans = None

npys = [np.load(f'{feature_dir}/{name}') for name in files]
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)

if big_npy.shape[0] > 2e5 and MiniBatchKMeans is not None:
    print('KMeans reduction:', big_npy.shape)
    big_npy = MiniBatchKMeans(
        n_clusters=10000, verbose=True, batch_size=256,
        compute_labels=False, init='random'
    ).fit(big_npy).cluster_centers_

np.save(f'{exp_dir}/total_fea.npy', big_npy)
n_ivf = max(1, min(int(16 * np.sqrt(big_npy.shape[0])), max(1, big_npy.shape[0] // 39)))
print('Index shape:', big_npy.shape, 'n_ivf:', n_ivf)

index = faiss.index_factory(256 if model_architecture == 'v1' else 768, f'IVF{n_ivf},Flat')
index_ivf = faiss.extract_index_ivf(index)
index_ivf.nprobe = 1
print('Training index...')
index.train(big_npy)
faiss.write_index(index, f'{exp_dir}/trained_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index')
print('Adding vectors...')
for i in range(0, big_npy.shape[0], 8192):
    index.add(big_npy[i:i+8192])
index_path = f'{exp_dir}/added_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index'
faiss.write_index(index, index_path)
print('Index ready:', index_path)


In [ ]:
#@title 16. Export model and index to RVC_Output
import shutil
from datetime import datetime

log_dir = Path(EXP_DIR)
model_files = sorted(log_dir.glob("*.pth"), key=lambda item: item.stat().st_mtime)
index_files = sorted(log_dir.glob("*.index"), key=lambda item: item.stat().st_mtime)
if not model_files or not index_files:
    raise FileNotFoundError("Model .pth or .index is missing. Run training and index creation first.")

source_model, source_index = model_files[-1], index_files[-1]
destination_model = Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.pth"
destination_index = Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.index"
if destination_model.exists() or destination_index.exists():
    archive = Path(DRIVE_OUTPUT_DIR) / "archive" / MODEL_NAME / datetime.now().strftime("%Y%m%d_%H%M%S")
    archive.mkdir(parents=True, exist_ok=True)
    for existing in (destination_model, destination_index):
        if existing.exists():
            shutil.copy2(existing, archive / existing.name)
    print(f"Previous export archived: {archive}")

shutil.copy2(source_model, destination_model)
shutil.copy2(source_index, destination_index)
if destination_model.stat().st_size == 0 or destination_index.stat().st_size == 0:
    raise RuntimeError("Export verification failed: an output file is empty.")
print(f"Exported: {destination_model}")
print(f"Exported: {destination_index}")


In [ ]:
#@title 17. Auto-disconnect runtime after successful export
from pathlib import Path
from google.colab import runtime
import time

required_files = [
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.pth",
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.index",
]

missing = [str(path) for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Export incomplet. Le runtime reste connecte.\nManquant:\n" + "\n".join(missing)
    )

print("Export confirme:")
for path in required_files:
    print(path)

print("Deconnexion du runtime dans 10 secondes...")
time.sleep(10)

runtime.unassign()
